# Figure 10

Cross-architecture spike comparison from the notebook path. The notebook runs the architecture experiment directly instead of shelling out to the CLI.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
from IPython.display import Image, display
import torch

arch = load_module('fig10_arch_run', 'experiments/archgrid/run.py')

device = torch.device('cpu')
archs = ['Transformer', 'MLP+LN', 'CNN+BN']
seeds = [0]
steps = 30
base_lr = 0.015
spike_at = 10
spike_mul = 10.0

data = arch.run_experiment(archs, seeds, steps, base_lr, spike_at, spike_mul, device)

out_dir = OUTPUT_ROOT / 'fig10'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'archgrid.png'
out_pdf = out_dir / 'archgrid.pdf'

arch.make_plot(data, archs, steps, spike_at, out_png, out_pdf, spike_mul)
display(Image(filename=str(out_png)))
arch.build_summary(
    data,
    archs,
    {
        'seeds': seeds,
        'archs': archs,
        'steps': steps,
        'base_lr': base_lr,
        'spike_at': spike_at,
        'spike_mul': spike_mul,
    },
    out_dir / 'archgrid_multiseed.pt',
    out_png,
    out_pdf,
    out_dir / 'summary.json',
)
